1. Imports y configuración global

In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from jiwer import cer, wer

warnings.filterwarnings("ignore")

DATASET_DIR = Path("../../data/processed/dataset/")
MANIFEST_PATH = DATASET_DIR / "manifest.json"

# Config
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
WARMUP_RUNS  = 3 
EVAL_SAMPLE  = None  

print(f"Dispositivo : {DEVICE}")
print(f"PyTorch     : {torch.__version__}")

# Cargar manifest
with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)

manifest = [m for m in manifest if m["label"].strip()]

if EVAL_SAMPLE:
    import random; random.seed(42)
    manifest = random.sample(manifest, EVAL_SAMPLE)

print(f"Muestras a evaluar: {len(manifest)}")

2. Utilidades: métricas y normalización de texto

In [ ]:
import unicodedata

def normalize_text(text: str) -> str:
    """
    Normalización mínima para comparación justa entre modelos:
    - Minúsculas
    - Strip de espacios extremos
    - Colapsar espacios múltiples
    NO elimina tildes ni ñ, para no penalizar/favorecer modelos.
    """
    text = text.lower().strip()
    text = " ".join(text.split())
    return text


def compute_metrics(predictions: list[str], references: list[str]) -> dict:
    """
    Calcula CER, WER y Accuracy a nivel de secuencia completa.
    predictions y references deben estar ya normalizados.
    """
    cer_score = cer(references, predictions)
    wer_score = wer(references, predictions)

    exact = sum(p == r for p, r in zip(predictions, references))
    acc   = exact / len(references) if references else 0.0

    return {
        "CER"      : round(cer_score, 4),
        "WER"      : round(wer_score, 4),
        "Accuracy" : round(acc, 4),
        "N"        : len(references),
    }


def load_image_rgb(path: str) -> Image.Image:
    """Carga imagen como RGB, compatible con todos los modelos."""
    return Image.open(path).convert("RGB")

3. Definición de los 4 modelos

In [ ]:
# Cada modelo expone la misma interfaz:
#   load()   → carga pesos en memoria
#   predict(image: PIL.Image) → str
#   name     → str identificador

class TrOCRModel:
    """Wrapper genérico para modelos HuggingFace TrOCR."""
    def __init__(self, model_id: str, name: str):
        self.model_id = model_id
        self.name     = name
        self.processor = None
        self.model     = None

    def load(self):
        print(f"  Cargando {self.name} ({self.model_id})...")
        self.processor = TrOCRProcessor.from_pretrained(self.model_id)
        self.model     = VisionEncoderDecoderModel.from_pretrained(self.model_id)
        self.model.to(DEVICE).eval()

    def predict(self, image: Image.Image) -> str:
        pixel_values = self.processor(
            images=image, return_tensors="pt"
        ).pixel_values.to(DEVICE)
        with torch.no_grad():
            ids = self.model.generate(pixel_values)
        return self.processor.batch_decode(ids, skip_special_tokens=True)[0]


class PARSeqMultilingualModel:
    """Wrapper para Felix92/doctr-torch-parseq-multilingual-v1 via docTR."""
    name = "PARSeq-Multilingual"

    def load(self):
        print(f"  Cargando {self.name}...")
        import os; os.environ["USE_TORCH"] = "1"
        from doctr.models import from_hub, recognition_predictor
        reco_arch  = from_hub("Felix92/doctr-torch-parseq-multilingual-v1")
        self._predictor = recognition_predictor(
            arch=reco_arch, pretrained=False
        )

    def predict(self, image: Image.Image) -> str:
        import numpy as np
        img_np = np.array(image)      
        result = self._predictor([img_np])
        return result[0][0] if result else ""
    

# Instanciar los 4 modelos
MODELS = [
    TrOCRModel(
        model_id = "microsoft/trocr-large-handwritten",
        name     = "TrOCR-Large-EN"
    ),
    TrOCRModel(
        model_id = "qantev/trocr-base-spanish",
        name     = "TrOCR-Base-ES"
    ),
    TrOCRModel(
        model_id = "Kansallisarkisto/multicentury-htr-model",
        name     = "Multicentury-HTR"
    ),
    PARSeqMultilingualModel(),
]

print("Modelos definidos:")
for m in MODELS:
    print(f"  • {m.name}")

4. Evaluación principal (loop por modelo)

In [ ]:
# Preparar ground truth
gt_labels = [normalize_text(m["label"]) for m in manifest]
img_paths = [str(DATASET_DIR / m["img_path"].replace("\\", "/")) for m in manifest]

all_results = {}

for model_obj in MODELS:
    print(f"\n{'='*60}")
    print(f"Evaluando: {model_obj.name}")
    print(f"{'='*60}")

    # Carga del modelo
    model_obj.load()

    # Warmup
    warmup_img = load_image_rgb(img_paths[0])
    for _ in range(WARMUP_RUNS):
        model_obj.predict(warmup_img)

    # Inferencia con medición de latencia
    predictions = []
    latencies   = []

    for i, path in enumerate(img_paths):
        img = load_image_rgb(path)

        t0  = time.perf_counter()
        pred = model_obj.predict(img)
        t1  = time.perf_counter()

        predictions.append(normalize_text(pred))
        latencies.append((t1 - t0) * 1000)   # ms

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(img_paths)}] "
                  f"último pred: '{pred[:40]}'")

    # Métricas
    metrics = compute_metrics(predictions, gt_labels)
    metrics["Latencia_media_ms"]  = round(np.mean(latencies), 2)
    metrics["Latencia_p50_ms"]    = round(np.percentile(latencies, 50), 2)
    metrics["Latencia_p95_ms"]    = round(np.percentile(latencies, 95), 2)

    all_results[model_obj.name] = {
        "metrics"     : metrics,
        "predictions" : predictions,
        "latencies"   : latencies,
    }

    print(f"\n  CER      : {metrics['CER']:.4f}")
    print(f"  WER      : {metrics['WER']:.4f}")
    print(f"  Accuracy : {metrics['Accuracy']:.4f}")
    print(f"  Latencia media: {metrics['Latencia_media_ms']} ms")

    #Liberar memoria entre modelos
    if hasattr(model_obj, "model") and model_obj.model is not None:
        del model_obj.model
    torch.cuda.empty_cache()

print("\n Evaluación completa.")

5. Tabla de resultados comparativa

In [ ]:
rows = []
for name, data in all_results.items():
    m = data["metrics"]
    rows.append({
        "Modelo"            : name,
        "CER ↓"             : m["CER"],
        "WER ↓"             : m["WER"],
        "Accuracy ↑"        : m["Accuracy"],
        "Latencia media ms ↓": m["Latencia_media_ms"],
        "Latencia p95 ms"   : m["Latencia_p95_ms"],
        "N muestras"        : m["N"],
    })

df_results = pd.DataFrame(rows).set_index("Modelo")

# Destacar el mejor valor por columna
def highlight_best(s):
    ascending = s.name in ["CER ↓", "WER ↓", "Latencia media ms ↓", "Latencia p95 ms"]
    best = s.min() if ascending else s.max()
    return ["font-weight: bold; color: green" if v == best else "" for v in s]

display(
    df_results.style
    .apply(highlight_best, subset=["CER ↓", "WER ↓", "Accuracy ↑",
                                    "Latencia media ms ↓", "Latencia p95 ms"])
    .format({"CER ↓": "{:.4f}", "WER ↓": "{:.4f}", "Accuracy ↑": "{:.4f}"})
    .set_caption("Comparación de modelos HTR — Sin fine-tuning")
)


csv_path = DATASET_DIR / "comparacion_modelos.csv"
df_results.to_csv(csv_path)
print(f"\nResultados guardados en: {csv_path}")

6. Visualización gráfica

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

nombres  = list(all_results.keys())
colores  = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

# CER y WER
ax1 = fig.add_subplot(gs[0, 0])
x   = np.arange(len(nombres))
w   = 0.35
ax1.bar(x - w/2, [all_results[n]["metrics"]["CER"] for n in nombres],
        w, label="CER", color=colores, alpha=0.85)
ax1.bar(x + w/2, [all_results[n]["metrics"]["WER"] for n in nombres],
        w, label="WER", color=colores, alpha=0.5, hatch="//")
ax1.set_xticks(x); ax1.set_xticklabels(nombres, rotation=15, ha="right")
ax1.set_title("CER y WER (↓ mejor)"); ax1.legend()
ax1.set_ylabel("Tasa de error")

# Accuracy
ax2 = fig.add_subplot(gs[0, 1])
acc_vals = [all_results[n]["metrics"]["Accuracy"] for n in nombres]
bars = ax2.bar(nombres, acc_vals, color=colores, alpha=0.85)
ax2.bar_label(bars, fmt="%.3f", padding=3)
ax2.set_xticklabels(nombres, rotation=15, ha="right")
ax2.set_title("Accuracy exacto (↑ mejor)")
ax2.set_ylim(0, 1.1); ax2.set_ylabel("Proporción")

# Latencia (boxplot)
ax3 = fig.add_subplot(gs[1, 0])
lat_data = [all_results[n]["latencies"] for n in nombres]
bp = ax3.boxplot(lat_data, labels=nombres, patch_artist=True, notch=False)
for patch, color in zip(bp["boxes"], colores):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax3.set_title("Distribución de latencia por imagen")
ax3.set_ylabel("ms"); ax3.tick_params(axis="x", rotation=15)

#Latencia media + p95
ax4 = fig.add_subplot(gs[1, 1])
lat_mean = [all_results[n]["metrics"]["Latencia_media_ms"] for n in nombres]
lat_p95  = [all_results[n]["metrics"]["Latencia_p95_ms"]  for n in nombres]
ax4.plot(nombres, lat_mean, "o-", label="Media",  color="#4C72B0", lw=2)
ax4.plot(nombres, lat_p95,  "s--", label="P95",   color="#DD8452", lw=2)
ax4.set_title("Latencia media vs P95 (ms)")
ax4.set_ylabel("ms"); ax4.legend()
ax4.tick_params(axis="x", rotation=15)

plt.suptitle("Comparación de modelos HTR preentrenados — Sin fine-tuning",
             fontsize=13, y=1.01)
plt.savefig(DATASET_DIR / "comparacion_modelos.png",
            bbox_inches="tight", dpi=150)
plt.show()
print("Gráfico guardado.")

7. Análisis cualitativo: muestra de predicciones

In [ ]:
# Ver ejemplos concretos donde los modelos coinciden o divergen
N_SHOW = 10
indices = list(range(min(N_SHOW, len(manifest))))

rows_q = []
for i in indices:
    row = {
        "Ground Truth": gt_labels[i],
        "Etiqueta"    : manifest[i]["etiqueta"],
    }
    for name, data in all_results.items():
        pred = data["predictions"][i]
        ok   = "OK" if pred == gt_labels[i] else "NOT OK"
        row[name] = f"{ok} {pred}"
    rows_q.append(row)

df_qual = pd.DataFrame(rows_q)
display(df_qual.style.set_properties(**{"text-align": "left"})
               .set_caption("Análisis cualitativo — primeras muestras"))

8. Análisis por tipo de etiqueta

In [ ]:
# ¿Qué campos son más difíciles para cada modelo?
etiquetas_unicas = sorted(set(m["etiqueta"] for m in manifest))

rows_et = []
for etq in etiquetas_unicas:
    idx = [i for i, m in enumerate(manifest) if m["etiqueta"] == etq]
    if len(idx) < 2:
        continue
    row = {"Etiqueta": etq, "N": len(idx)}
    for name, data in all_results.items():
        preds = [data["predictions"][i] for i in idx]
        refs  = [gt_labels[i]          for i in idx]
        row[f"CER_{name}"] = round(cer(refs, preds), 3)
    rows_et.append(row)

df_etq = pd.DataFrame(rows_et).set_index("Etiqueta")
display(df_etq.style.background_gradient(cmap="RdYlGn_r", axis=None,
                                          subset=[c for c in df_etq.columns
                                                  if c.startswith("CER")])
               .set_caption("CER por tipo de campo — cada modelo"))

9. Conclusión automática

In [ ]:
# Seleccionar el mejor modelo basándose en CER
mejor_cer    = df_results["CER ↓"].idxmin()
mejor_lat    = df_results["Latencia media ms ↓"].idxmin()
mejor_acc    = df_results["Accuracy ↑"].idxmax()

print("=" * 55)
print("RESUMEN DE COMPARACIÓN")
print("=" * 55)
print(f"  Mejor CER      : {mejor_cer}")
print(f"  Mejor Accuracy : {mejor_acc}")
print(f"  Menor latencia : {mejor_lat}")
print()

if mejor_cer == mejor_acc:
    print(f"   Modelo recomendado para fine-tuning: {mejor_cer}")
    print(f"     (lidera tanto en CER como en Accuracy)")
else:
    print(f"    Hay trade-off:")
    print(f"     - Para precisión: {mejor_cer} (CER: {df_results.loc[mejor_cer, 'CER ↓']})")
    print(f"     - Para velocidad: {mejor_lat} "
          f"({df_results.loc[mejor_lat, 'Latencia media ms ↓']} ms)")
    print(f"  → En HTR el CER es la métrica principal.")
    print(f"     Se recomienda continuar con: {mejor_cer}")

print()
print("NOTA: Estos resultados son SIN fine-tuning.")
print("El modelo ganador se usará como base para la siguiente fase.")